# Projeto de IA Médica: Arquitetura Dual-Stream
**Estratégia:** Undersampling + Redes Paralelas (RGB e Espectral) + Threshold de 85%.

In [ ]:
!pip install datasets tensorflow pandas matplotlib seaborn scikit-learn scikit-image -q
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from skimage.color import rgb2lab
from sklearn.model_selection import train_test_split
import gc

print("--- Inicializando Estratégia ---")
strategy = tf.distribute.get_strategy()
IMG_SIZE = (224, 224)
BATCH_SIZE = 16 * strategy.num_replicas_in_sync

In [ ]:
def dual_stream_model():
    input_rgb = layers.Input(shape=(224, 224, 3), name='rgb_input')
    input_spec = layers.Input(shape=(224, 224, 2), name='spectral_input')
    
    # Ramo RGB
    x1 = layers.Conv2D(32, 3, activation='relu')(input_rgb)
    x1 = layers.GlobalAveragePooling2D()(x1)
    
    # Ramo Espectral
    x2 = layers.Conv2D(32, 3, activation='relu')(input_spec)
    x2 = layers.GlobalAveragePooling2D()(x2)
    
    combined = layers.Concatenate()([x1, x2])
    output = layers.Dense(7, activation='softmax')(combined)
    return models.Model(inputs=[input_rgb, input_spec], outputs=output)

with strategy.scope():
    model = dual_stream_model()
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Modelo pronto!')